### Head of states (2016-2023)
From WhoGov dataset. 

In [ ]:
import pandas as pd

leaders_df = pd.read_csv("../data/LeadersAndParties/WhoGov_within_V3.1.csv")
len(leaders_df)

In [ ]:
leaders_df = leaders_df[leaders_df["year"] >= 2016]
print(len(leaders_df))

In [ ]:
leaders_df = leaders_df.sort_values('year', ascending=False).drop_duplicates(subset=['name', 'country_isocode', 'birthyear'], keep='first')
print(len(leaders_df))

In [ ]:
# Putin's partyfacts_id is missing as he often runs as an independent, but we know it's United Russia (2256)
leaders_df.loc[leaders_df["name"] == "Vladimir Vladimirovich Putin", ["partyfacts_id", "party"]] = [2256, "er"]
leaders_df[leaders_df["name"] == "Vladimir Vladimirovich Putin"]
len(leaders_df)

In [ ]:
leaders_df = leaders_df[~leaders_df["party"].str.lower().isin(["independent", "unknown"])]
print(len(leaders_df))

### Party Data Labels

In [ ]:
import pandas as pd

parties_df = pd.read_csv("../data/LeadersAndParties/V-Dem-CPD-Party-V2.csv")
print(len(parties_df))

### Merge

In [ ]:
import pandas as pd
#  V-Dem: pf_party_id
# who gov: partyfacts_id
# Optional: normalize strings

# Sort by year for merge_asof
parties_sorted = parties_df.sort_values("year").copy()
leaders_sorted = leaders_df.sort_values("year").copy()

merged_list = []

# Group by normalized party + country
for (party, country), group_leaders in leaders_sorted.groupby(["partyfacts_id", "country_isocode"]):
    party = int(party)

    group_parties = parties_sorted[
        (parties_sorted["pf_party_id"] == party) &
        (parties_sorted["country_text_id"] == country)
    ].sort_values("year")
   
    if not group_parties.empty:
        merged_group = pd.merge_asof(
            group_leaders,
            group_parties,
            left_on="year",
            right_on="year",
            direction="nearest",
            suffixes=("_leader", "_party")
        )
        merged_list.append(merged_group)
# Combine all groups
merged_closest = pd.concat(merged_list, ignore_index=True)
len(merged_closest)

In [ ]:
fixed_rows = []
fallback_cases = []
# Sort by year for merge_asof
parties_sorted = parties_df.sort_values("year").copy()
leaders_sorted = leaders_df.sort_values("year").copy()
columns_to_check = ["v2paclient_osp", "v2paculsup_osp", "v2paimmig_osp", "v2palgbt_osp", "v2parelig_osp", "v2pawomlab_osp", "v2xpa_antiplural", "v2xpa_popul"]

merged_list = []
for _, row in merged_closest.iterrows():
    if row[columns_to_check].isna().any():
        party = row["partyfacts_id"]
        country = row["country_isocode"]
        year = row["year"]

        # candidate rows for same party/country
        candidates = parties_sorted[
            (parties_sorted["pf_party_id"] == party) &
            (parties_sorted["country_text_id"] == country)
        ].copy()

        # compute distance in years
        candidates["year_diff"] = (candidates["year"] - year).abs()

        # sort by closeness
        candidates = candidates.sort_values("year_diff")

        # drop rows where womlab is NaN
        candidates_valid = candidates[~candidates["v2pawomlab"].isna()]

        if not candidates_valid.empty:
            replacement = candidates_valid.iloc[0]

            # replace party columns in row
            for col in parties_df.columns:
                row[col] = replacement[col]

            fallback_cases.append({
                "leader": row.get("Name", None),
                "party": party,
                "country": country,
                "original_year": year,
                "replacement_year": replacement["year"],
            })

    fixed_rows.append(row)

merged_fixed = pd.DataFrame(fixed_rows)
merged_closest = merged_fixed
len(merged_closest)

## Keep only columns that are useful and make flags Boolean

In [ ]:
merged_closest.head()

leaders_final = merged_closest[["name", "year", "country_isocode", "position", "gender", "party", "party_english", "partyfacts_id", "birthyear", "e_regiongeo", "e_regionpol", "e_regionpol_6C",
                                    "v2xpa_antiplural",
                                    "v2xpa_popul",
                                    "v2paimmig", "v2paimmig_osp",   # Immigration
                                    "v2palgbt", "v2palgbt_osp",     # LGBT
                                    "v2paculsup", "v2paculsup_osp", # Cultural superiority 
                                    "v2parelig", "v2parelig_osp",   # Welfare
                                    "v2pawomlab", "v2pawomlab_osp", # Working Women 
                                    "v2paclient", "v2paclient_osp", # Clientelism
                                    "v2pariglef", "v2pariglef_osp"  # Left right
                                ]]

In [ ]:
flag_cols = [
    "v2xpa_antiplural",
    "v2xpa_popul",
    "v2paimmig", "v2paimmig_osp",   # Immigration
    "v2palgbt", "v2palgbt_osp",     # LGBT
    "v2paculsup", "v2paculsup_osp", # Cultural superiority 
    "v2parelig", "v2parelig_osp",   # Welfare
    "v2pawomlab", "v2pawomlab_osp", # Working Women 
    "v2paclient", "v2paclient_osp", # Clientelism
    "v2pariglef", "v2pariglef_osp" 
]

leaders_final = leaders_final.copy()

for col in flag_cols:
    if col.endswith("osp"):
        threshold = 2
    elif col.startswith("v2xpa"):
        threshold = 0.5

    leaders_final[f"{col}_bool"] = leaders_final[col].gt(threshold).astype("int8")

In [ ]:
leaders_final.to_csv("../data/LeadersAndParties/leaders.csv", index=False)


### Parties

In [ ]:
parties_df = parties_df[parties_df["year"] >= 2016]
unique_parties = (
    parties_df
    .sort_values("year", ascending=False)
    .drop_duplicates(
        subset=["v2paenname", "country_text_id"],
        keep="first"
    )
    .reset_index(drop=True)
)
print(len(unique_parties))

In [ ]:
unique_parties = unique_parties[
    unique_parties["v2paseatshare"] > 10
].reset_index(drop=True)
print(len(unique_parties))

In [ ]:
unique_parties.to_csv("../data/LeadersAndParties/parties.csv", index=False)
